# Meme Coin Fraud Detection — Agent Workflow

Implements the full autonomous pipeline for Meme Coin Early Warning System:

| Step | Tool | Description |
|------|------|-------------|
| 1 | `fetch_transactions` | MCP-style: pull raw on-chain records via Etherscan |
| 2 | `extract_graph_features` | NetworkX: convert transactions → graph metrics |
| 3 | `predict_fraud_probability` | CatBoost model: produce ML fraud score (0-100%) |
| 4 | `analyze_contract_code` | AST/regex: scan Solidity source for vulnerability patterns |
| 5 | Synthesis | Agent combines scores: **70% ML + 30% AST** → final Risk Level |

The LLM (via DeepSeek) acts as the orchestrator and decides tool call order autonomously.

In [1]:
import sys, os, json, time
from pathlib import Path
from openai import RateLimitError

# Add tools directory to path so imports work inside the notebook
TOOLS_DIR = Path.cwd() / "tools"
sys.path.insert(0, str(TOOLS_DIR))

from fetch_tool import fetch_transactions
from graph_tool import extract_graph_features
from predict_tool import predict_fraud_probability
from ast_tool import analyze_contract_code

print("All tools loaded successfully.")
print(f"Tools directory: {TOOLS_DIR}")

All tools loaded successfully.
Tools directory: c:\WorkSpace\Code\meme_coin_early_warning_system\03_Agents_and_Tools\tools


In [2]:
from openai import OpenAI
from dotenv import load_dotenv

# Load .env from workspace root (searches parent directories automatically)
load_dotenv()

DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")
if not DEEPSEEK_API_KEY:
    raise EnvironmentError(
        "DEEPSEEK_API_KEY not found. Add it to the .env file in the workspace root."
    )

MODEL = "deepseek-chat"
client = OpenAI(
    base_url="https://api.deepseek.com",
    api_key=DEEPSEEK_API_KEY,
)

print(f"DeepSeek client configured -> model: {MODEL}")


DeepSeek client configured -> model: deepseek-chat


In [3]:
# Maps tool name -> callable
TOOL_FUNCTIONS = {
    "fetch_transactions":       fetch_transactions,
    "extract_graph_features":   extract_graph_features,
    "predict_fraud_probability": predict_fraud_probability,
    "analyze_contract_code":    analyze_contract_code,
}

# OpenAI function-calling schemas
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "fetch_transactions",
            "description": (
                "Fetch ERC-20 token transfer records for a contract address "
                "from the Etherscan blockchain API."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "address":  {"type": "string", "description": "The ERC-20 contract address"},
                    "chain_id": {"type": "string", "description": "'1' = Ethereum, '56' = BSC", "default": "1"},
                },
                "required": ["address"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "extract_graph_features",
            "description": (
                "Convert a raw transaction list into NetworkX graph metrics "
                "(max_centrality, avg_clustering, unique_wallets, value_volatility, tx_count)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "transactions": {
                        "type": "array",
                        "description": "List of transaction dicts from fetch_transactions",
                    }
                },
                "required": ["transactions"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "predict_fraud_probability",
            "description": (
                "Load the pre-trained CatBoost model and calculate a fraud probability "
                "(0–100%) from the five graph features."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "features": {
                        "type": "object",
                        "description": "Dict with keys: max_centrality, avg_clustering, unique_wallets, value_volatility, tx_count",
                        "properties": {
                            "max_centrality":   {"type": "number"},
                            "avg_clustering":   {"type": "number"},
                            "unique_wallets":   {"type": "number"},
                            "value_volatility": {"type": "number"},
                            "tx_count":         {"type": "number"},
                        },
                    }
                },
                "required": ["features"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_contract_code",
            "description": (
                "Fetch the Solidity source code from Etherscan and scan it for "
                "vulnerability patterns: hidden mints, owner backdoors, blacklists, "
                "trading-pause functions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "address": {"type": "string", "description": "The contract address"},
                },
                "required": ["address"],
            },
        },
    },
]

print(f"{len(TOOLS)} tools registered.")

4 tools registered.


In [4]:
def run_agent(contract_address: str, verbose: bool = True) -> dict:
    """
    Autonomous fraud detection agent.
    Final score = ML_probability * 0.70 + AST_score_pct * 0.30
    Thresholds balance Recall and Precision: HIGH >= 70, MEDIUM >= 50.
    """
    system_prompt = """You are a blockchain forensics agent that detects meme coin rug pulls.

Workflow — call all four tools in order, then produce a final report:
1. fetch_transactions(address)          -> get on-chain records
2. extract_graph_features(transactions) -> compute graph metrics
3. predict_fraud_probability(features)  -> get ML fraud score (0-100%)
4. analyze_contract_code(address)       -> get AST risk score (0.0-1.0) + findings

After all tools complete, output a structured JSON report:
{
  "contract_address": "...",
  "ml_score": "<value>%",
  "ast_score": "<value>%",
  "final_risk_score": "<value>%",
  "risk_level": "HIGH RISK | MEDIUM RISK | LOW RISK",
  "ast_findings": [...],
  "explanation": "..."
}

Formula: final_score = (ml_score * 0.70) + (ast_score * 100 * 0.30)

Risk levels (balanced Recall-Precision):
  >= 70 -> HIGH RISK  |  >= 50 -> MEDIUM RISK  |  < 50 -> LOW RISK

Escalation rules — apply in order:
1. HIGH RISK if final_score >= 70 AND ml_score >= 65 (both signals must be elevated)
2. HIGH RISK if ml_score >= 80 regardless of AST (strong ML signal alone is sufficient)
3. MEDIUM RISK if final_score >= 50
4. LOW RISK otherwise
Do NOT assign HIGH RISK based on AST patterns alone when ml_score < 50% — many
legitimate tokens share ownership/mint/pause patterns with rug pulls.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": f"Analyze this contract for rug pull risk: {contract_address}"},
    ]

    tool_results = {}

    for iteration in range(12):  # safety cap
        for attempt in range(3):
            try:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=messages,
                    tools=TOOLS,
                    tool_choice="auto",
                )
                break
            except RateLimitError as e:
                if attempt == 2:
                    raise
                wait = 35
                try:
                    wait = e.response.json()["error"]["metadata"]["retry_after_seconds"] + 5
                except Exception:
                    pass
                print(f"  [Rate limit] Waiting {wait:.0f}s before retry {attempt + 2}/3...")
                time.sleep(wait)

        choice = response.choices[0]
        msg    = choice.message

        if verbose:
            print(f"[Round {iteration + 1}] finish_reason={choice.finish_reason}")

        if not msg.tool_calls:
            print("\n" + "=" * 60)
            print("FINAL AGENT REPORT")
            print("=" * 60)
            print(msg.content)
            return {"final_response": msg.content, "tool_results": tool_results}

        messages.append(msg)

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)

            if verbose:
                print(f"  → {fn_name}({list(fn_args.keys())})")

            # extract_graph_features: always use the FULL stored transactions,
            # not the truncated version the LLM re-passed in its tool call.
            if fn_name == "extract_graph_features" and "fetch_transactions" in tool_results:
                full_txns = tool_results["fetch_transactions"].get("transactions", [])
                result = TOOL_FUNCTIONS["extract_graph_features"](transactions=full_txns)
            else:
                result = TOOL_FUNCTIONS[fn_name](**fn_args)

            tool_results[fn_name] = result

            result_str = json.dumps(result)
            if len(result_str) > 4000 and fn_name == "fetch_transactions":
                result_str = json.dumps({
                    "success": result.get("success"),
                    "count":   result.get("count"),
                    "transactions": result.get("transactions", [])[:5],
                    "note": f"Truncated — showing 5 of {result.get('count')} transactions",
                })

            messages.append({
                "role":         "tool",
                "tool_call_id": tc.id,
                "content":      result_str,
            })

    return {"final_response": "Max iterations reached.", "tool_results": tool_results}


In [5]:
# Demo
# Replace with any ERC-20 contract address needs to be audited.
# Known fraud example:
FRAUD_ADDRESS = "0x21df3b628c2594c18c6f488d22b574f5a33e3fb4"

# Known safe example (Uniswap UNI token):
# SAFE_ADDRESS  = "0x1f9840a85d5aF5bf1D1762F925BDADdC4201F984"

print(f"Auditing contract: {FRAUD_ADDRESS}")
print("Ensure OPENROUTER_API_KEY is set in Cell 3 before running.")
print("-" * 60)

result = run_agent(FRAUD_ADDRESS)

Auditing contract: 0x21df3b628c2594c18c6f488d22b574f5a33e3fb4
Ensure OPENROUTER_API_KEY is set in Cell 3 before running.
------------------------------------------------------------
[Round 1] finish_reason=tool_calls
  → fetch_transactions(['address', 'chain_id'])
[Round 2] finish_reason=tool_calls
  → extract_graph_features(['transactions'])
[Round 3] finish_reason=tool_calls
  → predict_fraud_probability(['features'])


c:\WorkSpace\Code\meme_coin_early_warning_system\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[Round 4] finish_reason=tool_calls
  → analyze_contract_code(['address'])
[Round 5] finish_reason=stop

FINAL AGENT REPORT
Now I have all the data. Let me compile the final report.

**Calculations:**

- **ml_score**: 99.95%
- **ast_score**: 0.35 → 35%
- **final_score** = (99.95 * 0.70) + (35 * 0.30) = 69.965 + 10.5 = **80.465%**

**Escalation Rules Check:**
1. Rule 1: final_score >= 70 AND ml_score >= 65 → 80.465% ≥ 70 ✓ AND 99.95% ≥ 65 ✓ → **HIGH RISK**
2. Rule 2: ml_score >= 80 → 99.95% ≥ 80 ✓ → **HIGH RISK** (also applies)

```json
{
  "contract_address": "0x21df3b628c2594c18c6f488d22b574f5a33e3fb4",
  "ml_score": "99.95%",
  "ast_score": "35%",
  "final_risk_score": "80.47%",
  "risk_level": "HIGH RISK",
  "ast_findings": [
    "Public Mint Function detected in _mint - minting capability allows unlimited token creation (risk_score: 0.35)"
  ],
  "explanation": "This token (NFTify/N1) exhibits extremely strong signals of a rug pull scam. The ML fraud detection model assigned a 99.95

# Test Suite — Recall Validation

Run the agent against a curated set of **known-fraud** and **known-safe** contracts to validate that the Recall-optimised pipeline correctly flags rug pulls.

> **Note on test case selection:** This model is trained on *meme coin* transaction graphs. Major DeFi protocols (USDC, USDT, UNI) have tx_counts in the tens of millions — orders of magnitude above the training distribution — causing meaningless OOD scores. Safe cases below are established meme coins that never rug-pulled and are more in-domain.

| # | Address | Ground Truth | Notes |
|---|---------|-------------|-------|
| 1 | `0x21df3b628c2594c18c6f488d22b574f5a33e3fb4` | FRAUD | Known rug pull |
| 2 | `0x00e25f0fd5d6a36c252398ca14f3f29128cf5bf9` | FRAUD | Training set fraud |
| 3 | `0x010496d8f3269ee9b05060bdf40c2d66e83e61dd` | FRAUD | Training set fraud |
| 4 | `0x95aD61b0a150d79219dCF64E1E6Cc01f0B64C4cE` | SAFE | Shiba Inu (SHIB) — survived since 2020 |
| 5 | `0x6982508145454Ce325dDbE47a25d4ec3d2311933` | SAFE | PEPE — launched 2023, still active |
| 6 | `0xcf0C122c6b73ff809C693DB761e7BaeBe62b6a2E` | SAFE | FLOKI — established meme coin |

Run the cell below to execute all cases sequentially (5-second delay between calls to respect API rate limits).


In [6]:
# ── Test case registry ────────────────────────────────────────────────────────
TEST_CASES = [
    # Known rug pulls
    {"address": "0x21df3b628c2594c18c6f488d22b574f5a33e3fb4", "expected": "FRAUD"},
    {"address": "0x00e25f0fd5d6a36c252398ca14f3f29128cf5bf9", "expected": "FRAUD"},
    {"address": "0x010496d8f3269ee9b05060bdf40c2d66e83e61dd", "expected": "FRAUD"},
    # Established meme coins that never rug-pulled (in-domain safe examples)
    {"address": "0x95aD61b0a150d79219dCF64E1E6Cc01f0B64C4cE", "expected": "SAFE"},   # SHIB
    {"address": "0x6982508145454Ce325dDbE47a25d4ec3d2311933", "expected": "SAFE"},   # PEPE
    {"address": "0xcf0C122c6b73ff809C693DB761e7BaeBe62b6a2E", "expected": "SAFE"},   # FLOKI
]

# Set to a subset to save API quota, e.g. TEST_CASES[:2]
CASES_TO_RUN = TEST_CASES

print(f"Test suite: {len(CASES_TO_RUN)} contracts queued")
for i, c in enumerate(CASES_TO_RUN, 1):
    print(f"  [{i}] {c['address']}  (expected: {c['expected']})")


Test suite: 6 contracts queued
  [1] 0x21df3b628c2594c18c6f488d22b574f5a33e3fb4  (expected: FRAUD)
  [2] 0x00e25f0fd5d6a36c252398ca14f3f29128cf5bf9  (expected: FRAUD)
  [3] 0x010496d8f3269ee9b05060bdf40c2d66e83e61dd  (expected: FRAUD)
  [4] 0x95aD61b0a150d79219dCF64E1E6Cc01f0B64C4cE  (expected: SAFE)
  [5] 0x6982508145454Ce325dDbE47a25d4ec3d2311933  (expected: SAFE)
  [6] 0xcf0C122c6b73ff809C693DB761e7BaeBe62b6a2E  (expected: SAFE)


In [7]:
import re

# ── Batch runner ──────────────────────────────────────────────────────────────
batch_results = []

for idx, case in enumerate(CASES_TO_RUN):
    addr     = case["address"]
    expected = case["expected"]
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(CASES_TO_RUN)}] {addr}  (expected: {expected})")
    print("="*60)

    try:
        res        = run_agent(addr, verbose=True)
        raw        = res.get("final_response", "")
        tool_res   = res.get("tool_results", {})

        # ── Parse final report ────────────────────────────────────
        risk_level = "UNKNOWN"
        m = re.search(r'"risk_level"\s*:\s*"([^"]+)"', raw)
        if m:
            risk_level = m.group(1).strip()

        score = None
        m2 = re.search(r'"final_risk_score"\s*:\s*"?([\d.]+)', raw)
        if m2:
            score = float(m2.group(1))

        # ── Extract component scores from tool results ────────────
        ml_prob  = tool_res.get("predict_fraud_probability", {}).get("ml_probability")
        ast_raw  = tool_res.get("analyze_contract_code",    {}).get("ast_risk_score")
        ast_pct  = round(ast_raw * 100, 1) if ast_raw is not None else None
        ast_findings = tool_res.get("analyze_contract_code", {}).get("findings", [])

        print(f"  ML score  : {ml_prob}%")
        print(f"  AST score : {ast_pct}%  findings: {[f.get('issue') for f in ast_findings]}")
        print(f"  Final     : {score}%  → {risk_level}")

        correct = (
            ("FRAUD" in expected and "HIGH" in risk_level) or
            ("SAFE"  in expected and "LOW"  in risk_level) or
            ("SAFE"  in expected and "MEDIUM" in risk_level and score is not None and score < 50)
        )

        batch_results.append({
            "address":   addr,
            "expected":  expected,
            "predicted": risk_level,
            "ml%":       f"{ml_prob}%" if ml_prob is not None else "N/A",
            "ast%":      f"{ast_pct}%" if ast_pct is not None else "N/A",
            "final%":    f"{score:.1f}%" if score is not None else "N/A",
            "correct":   "✓" if correct else "✗",
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        batch_results.append({
            "address": addr, "expected": expected,
            "predicted": "ERROR", "ml%": "N/A", "ast%": "N/A",
            "final%": "N/A", "correct": "✗",
        })

    if idx < len(CASES_TO_RUN) - 1:
        time.sleep(5)

print("\n\nAll cases completed.")



[1/6] 0x21df3b628c2594c18c6f488d22b574f5a33e3fb4  (expected: FRAUD)
[Round 1] finish_reason=tool_calls
  → fetch_transactions(['address', 'chain_id'])
  → analyze_contract_code(['address'])
[Round 2] finish_reason=tool_calls
  → extract_graph_features(['transactions'])
[Round 3] finish_reason=tool_calls
  → predict_fraud_probability(['features'])


c:\WorkSpace\Code\meme_coin_early_warning_system\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[Round 4] finish_reason=stop

FINAL AGENT REPORT
Excellent! Let me now compile the final report.

## Data Summary

**ML Score**: 99.95%
- Extremely high fraud probability from the graph analysis model. The max_centrality of 0.97 is very high (one wallet dominates), and the value_volatility is enormous (3.47e25), indicating suspicious concentration and distribution patterns.

**AST Score**: 35% (0.35 × 100)
- Only finding is a public mint function (_mint), which by itself is not definitive proof of a rug pull — many legitimate tokens have mint capabilities.

**Final Score Calculation**:
- `final_score = (99.95 * 0.70) + (35 * 0.30) = 69.965 + 10.5 = 80.47%`

**Risk Level Determination** (escalation rules):
1. ✅ ml_score (99.95) >= 80 → **HIGH RISK** (strong ML signal alone is sufficient)

---

Here is the structured report:

```json
{
  "contract_address": "0x21df3b628c2594c18c6f488d22b574f5a33e3fb4",
  "ml_score": "99.95%",
  "ast_score": "35%",
  "final_risk_score": "80.47%",
  "risk_

c:\WorkSpace\Code\meme_coin_early_warning_system\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[Round 4] finish_reason=stop

FINAL AGENT REPORT
Excellent! All data is in. Let me compile the final analysis.

## Final Report

**Calculation:**
- **ML Score**: 99.94%
- **AST Score**: 0.35 → 35%
- **Final Score**: (99.94 × 0.70) + (35 × 0.30) = 69.96 + 10.5 = **80.46%**

**Escalation Check:**
1. ✅ final_score >= 70 AND ml_score >= 65 → **HIGH RISK** triggered
2. ✅ ml_score >= 80 (99.94) → **HIGH RISK** triggered regardless

```json
{
  "contract_address": "0x00e25f0fd5d6a36c252398ca14f3f29128cf5bf9",
  "ml_score": "99.94%",
  "ast_score": "35%",
  "final_risk_score": "80.46%",
  "risk_level": "HIGH RISK",
  "ast_findings": [
    {
      "issue": "Public Mint Function",
      "function": "_mint",
      "risk_score": 0.35,
      "detail": "Minting capability detected — allows token creation after launch"
    }
  ],
  "explanation": "This token (KWIK / Kwikswap) exhibits extremely strong on-chain indicators of a rug pull. The ML graph-based model scores it at 99.94% fraudulent probabili

...

[Round 4] finish_reason=stop

FINAL AGENT REPORT
Now I have all the data. Let me compile the report.

**Analysis Summary:**

1. **Contract**: `0x95aD61b0a150d79219dCF64E1E6Cc01f0B64C4cE` — This is the **SHIBA INU (SHIB)** token contract on Ethereum.
2. **ML Score**: 1.12% — very low fraud probability from graph analysis (419 unique wallets, 1000 txns, healthy decentralization)
3. **AST Score**: 0.0% — the contract code analysis found zero risky patterns
4. **Final Score**: (1.12 * 0.70) + (0.0 * 100 * 0.30) = **0.78%**

Let me apply the escalation rules:
- Rule 1: final_score >= 70 AND ml_score >= 65? No (0.78% and 1.12%)
- Rule 2: ml_score >= 80? No (1.12%)
- Rule 3: final_score >= 50? No (0.78%)
- Rule 4: **LOW RISK**

```json
{
  "contract_address": "0x95aD61b0a150d79219dCF64E1E6Cc01f0B64C4cE",
  "ml_score": "1.12%",
  "ast_score": "0.0%",
  "final_risk_score": "0.78%",
  "risk_level": "LOW RISK",
  "ast_findings": [],
  "explanation": "This is the legitimate SHIBA INU (SHIB) token 

c:\WorkSpace\Code\meme_coin_early_warning_system\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[Round 4] finish_reason=stop

FINAL AGENT REPORT
Now I have all the data. Let me compute the final score and produce the report.

**Calculations:**

- **ML Score:** 1.28%
- **AST Score:** 0.3 (30%)
- **Final Score:** (1.28 * 0.70) + (30 * 0.30) = 0.896 + 9.0 = **9.896%**

**Risk Level Assessment:**
- Final score (9.896%) < 50 → LOW RISK
- Check escalation: 
  1. Final >= 70 AND ML >= 65? No (9.9 < 70)
  2. ML >= 80? No (1.28 < 80)
  3. Final >= 50? No (9.9 < 50)
  4. → LOW RISK

This is the **Pepe (PEPE)** token, one of the most well-known meme coins on Ethereum. Let me compile the final report.

```json
{
  "contract_address": "0x6982508145454Ce325dDbE47a25d4ec3d2311933",
  "ml_score": "1.28%",
  "ast_score": "30.00%",
  "final_risk_score": "9.90%",
  "risk_level": "LOW RISK",
  "ast_findings": [
    "Dangerous Owner Privilege (blacklist function) — risk contribution 15%",
    "Blacklist Mechanism detected (blacklists variable) — risk contribution 15%"
  ],
  "explanation": "This is t

c:\WorkSpace\Code\meme_coin_early_warning_system\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[Round 4] finish_reason=stop

FINAL AGENT REPORT
All four tools have returned their results. Let me compute the final risk assessment.

**Data Summary:**

1. **ML Fraud Probability**: 1.29% (very low)
2. **AST Risk Score**: 0.0 (no concerning patterns found)
3. **Findings**: None - contract code has no hidden mints, owner backdoors, blacklists, or trading-pause functions detected.

**Final Score Calculation:**
- final_score = (1.29 * 0.70) + (0.0 * 100 * 0.30)
- final_score = 0.903 + 0.0 = **0.90%**

**Escalation Rules Check:**
1. final_score >= 70? No (0.90). ml_score >= 65? No. → Skip
2. ml_score >= 80? No (1.29). → Skip
3. final_score >= 50? No (0.90). → Skip
4. → **LOW RISK**

Here is the structured final report:

```json
{
  "contract_address": "0xcf0c122c6b73ff809c693db761e7baeBe62b6a2E",
  "ml_score": "1.29%",
  "ast_score": "0.0%",
  "final_risk_score": "0.90%",
  "risk_level": "LOW RISK",
  "ast_findings": [],
  "explanation": "This is the well-known FLOKI (FLOKI) token — a le

In [8]:
import pandas as pd

df_results = pd.DataFrame(batch_results)
print("\n" + "="*85)
print("RECALL VALIDATION SUMMARY  (ml% = CatBoost only | ast% = AST scanner only)")
print("="*85)
print(df_results.to_string(index=False))

total   = len(df_results)
correct = (df_results["correct"] == "✓").sum()
fraud_cases = df_results[df_results["expected"] == "FRAUD"]
safe_cases  = df_results[df_results["expected"] == "SAFE"]
recall_tp   = (fraud_cases["correct"] == "✓").sum()
fp_count    = (safe_cases["correct"] == "✗").sum()

print(f"\nFraud Recall (TP/TP+FN) : {recall_tp}/{len(fraud_cases)} ({recall_tp/max(len(fraud_cases),1)*100:.0f}%)")
print(f"False Positives (safe→HIGH): {fp_count}/{len(safe_cases)}")
print(f"\nNote: High AST scores on 'safe' meme coins are EXPECTED — SHIB/PEPE/FLOKI")
print(f"      genuinely contain mint/owner/blacklist patterns identical to rug pulls.")
print(f"      This is the Recall-Precision trade-off: 0 missed frauds, some false alarms.")



RECALL VALIDATION SUMMARY  (ml% = CatBoost only | ast% = AST scanner only)
                                   address expected predicted    ml%  ast% final% correct
0x21df3b628c2594c18c6f488d22b574f5a33e3fb4    FRAUD HIGH RISK 99.95% 35.0%  80.5%       ✓
0x00e25f0fd5d6a36c252398ca14f3f29128cf5bf9    FRAUD HIGH RISK 99.94% 35.0%  80.5%       ✓
0x010496d8f3269ee9b05060bdf40c2d66e83e61dd    FRAUD HIGH RISK 99.84% 35.0%  80.4%       ✓
0x95aD61b0a150d79219dCF64E1E6Cc01f0B64C4cE     SAFE  LOW RISK  1.12%  0.0%   0.8%       ✓
0x6982508145454Ce325dDbE47a25d4ec3d2311933     SAFE  LOW RISK  1.28% 30.0%   9.9%       ✓
0xcf0C122c6b73ff809C693DB761e7BaeBe62b6a2E     SAFE  LOW RISK  1.29%  0.0%   0.9%       ✓

Fraud Recall (TP/TP+FN) : 3/3 (100%)
False Positives (safe→HIGH): 0/3

Note: High AST scores on 'safe' meme coins are EXPECTED — SHIB/PEPE/FLOKI
      genuinely contain mint/owner/blacklist patterns identical to rug pulls.
      This is the Recall-Precision trade-off: 0 missed frauds, some fa